In [1]:
import pandas as pd
import numpy as np
import csv
import re

In [2]:
df1 = pd.read_csv("Baseline_EFs/rateperhouroni_smoke_17031_2022_OZONETOXICSPMGHGMETALS_17031_7.csv",skiprows=2, low_memory=False)
df2 = pd.read_csv("Baseline_EFs/rateperhouroni_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_7.csv",skiprows=2, low_memory=False)
df3 = pd.read_csv("FleetRenewal_EFs/rateperhouroni_smoke_17031_2022_OZONETOXICSPMGHGMETALS_17031_7.csv",skiprows=2, low_memory=False)

In [3]:
def _veh_code_from_agg_scc(series):
    s = series.astype(str).str.zfill(10)
    return s.str[4:6]  # 0-based index: positions 4 and 5 are the 5th/6th digits

In [4]:
align_keys = [
    'MOVESScenarioID','yearID','FIPS','monthID',
    'agg_scc','temperature',
]

df1['MOVESScenarioID'] = df1['MOVESScenarioID'].astype(str).str.strip().str.casefold()
df2['MOVESScenarioID'] = df2['MOVESScenarioID'].astype(str).str.strip().str.casefold()
df3['MOVESScenarioID'] = df3['MOVESScenarioID'].astype(str).str.strip().str.casefold()

for k in align_keys:
    assert k in df1.columns, f"df1 missing key column: {k}"
    assert k in df2.columns, f"df2 missing key column: {k}"
    assert k in df3.columns, f"df3 missing key column: {k}"

In [5]:
df1.loc[(df1['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["62"]))]

,MOVESScenarioID,yearID,monthID,FIPS,agg_scc,temperature,THC_INV,CO_INV,NOX,CH4_INV,...,FORM_PRIMARY,HONO,NAPHTH,NAPHTHALENE,NH3,NO,NO2,PMC,PMFINE,SO2
23,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,25,2.909348,18.305154,51.141120,0.391356,...,0.006500,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405
32,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,25,192.070293,90.560563,17.989520,184.191658,...,0.039748,0.003129,0.000070,5.481966e-07,0.026881,0.338275,0.049688,0.003814,0.014827,0.001013
47,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,25,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
71,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,30,2.909348,18.305155,51.141120,0.391356,...,0.006500,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405
80,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,30,192.070293,90.560563,17.989520,184.191658,...,0.039748,0.003129,0.000070,5.481966e-07,0.026881,0.338275,0.049688,0.003814,0.014827,0.001013
95,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,30,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
119,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,35,2.909348,18.305154,51.141120,0.391356,...,0.006500,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405
128,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,35,192.070293,90.560563,17.989520,184.191658,...,0.039748,0.003129,0.000070,5.481966e-07,0.026881,0.338275,0.049688,0.003814,0.014827,0.001013
143,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,35,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
167,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,40,2.909348,18.305155,51.141120,0.391356,...,0.006500,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096278,0.168404,0.000405


In [6]:
df2.loc[(df2['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["62"]))]

,MOVESScenarioID,yearID,monthID,FIPS,agg_scc,temperature,THC_INV,CO_INV,NOX,CH4_INV,...,FORM_PRIMARY,HONO,NAPHTH,NAPHTHALENE,NH3,NO,NO2,PMC,PMFINE,SO2
23,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,25,2.909348,18.305154,51.14112,0.391356,...,0.00650,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405
32,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,25,189.567993,90.513496,17.98880,181.792007,...,0.03923,0.003129,0.000069,5.392680e-07,0.026881,0.338261,0.049686,0.000000,0.000000,0.001013
47,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,25,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
71,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,30,2.909348,18.305154,51.14112,0.391356,...,0.00650,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405
80,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,30,189.567993,90.513496,17.98880,181.792007,...,0.03923,0.003129,0.000069,5.392680e-07,0.026881,0.338261,0.049686,0.000000,0.000000,0.001013
95,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,30,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
119,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,35,2.909348,18.305155,51.14112,0.391356,...,0.00650,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096277,0.168404,0.000405
128,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,35,189.567993,90.513496,17.98880,181.792007,...,0.03923,0.003129,0.000069,5.392680e-07,0.026881,0.338261,0.049686,0.000000,0.000000,0.001013
143,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,35,0.000000,0.000000,0.00000,0.000000,...,0.00000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
167,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,40,2.909348,18.305154,51.14112,0.391356,...,0.00650,0.008894,0.018435,1.438294e-04,0.020282,0.925167,0.177746,0.096276,0.168404,0.000405


In [7]:
df3.loc[(df3['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["62"]))]

,MOVESScenarioID,yearID,monthID,FIPS,agg_scc,temperature,THC_INV,CO_INV,NOX,CH4_INV,...,FORM_PRIMARY,HONO,NAPHTH,NAPHTHALENE,NH3,NO,NO2,PMC,PMFINE,SO2
23,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,25,1.234249,15.999708,39.614344,0.310970,...,0.000825,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381
32,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,25,191.835228,90.582876,17.968319,183.969758,...,0.039508,0.003125,0.000070,5.467572e-07,0.026713,0.337877,0.049629,0.003683,0.014293,0.001014
47,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,25,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
71,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,30,1.234249,15.999708,39.614345,0.310970,...,0.000825,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381
80,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,30,191.834237,90.582876,17.968319,183.969758,...,0.039508,0.003125,0.000070,5.467572e-07,0.026713,0.337877,0.049629,0.003683,0.014293,0.001014
95,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,30,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
119,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,35,1.234249,15.999708,39.614345,0.310970,...,0.000825,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381
128,rd_17031_2022_7_t25_115,2022,7,17031,2203620192,35,191.834237,90.582876,17.968319,183.969758,...,0.039508,0.003125,0.000070,5.467572e-07,0.026713,0.337877,0.049629,0.003683,0.014293,0.001014
143,rd_17031_2022_7_t25_115,2022,7,17031,2209620192,35,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000e+00,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
167,rd_17031_2022_7_t25_115,2022,7,17031,2202620192,40,1.234249,15.999708,39.614345,0.310970,...,0.000825,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381


<h1> Merge files 

In [8]:
merged_12 = pd.merge(
    df1, df2,
    on=align_keys,
    how='inner',
    suffixes=('_df1','_df2')
)

print("Merged df1 and df2:")
print("  df1 rows:", len(df1))
print("  df2 rows:", len(df2))
print("  merged rows (should be close to or equal to df1/df2):", len(merged_12))

numeric_cols_df1 = df1.select_dtypes(include=[np.number]).columns.tolist()
numeric_cols_df2 = df2.select_dtypes(include=[np.number]).columns.tolist()

key_like_cols = set(align_keys)
pollutant_cols_df1 = [c for c in numeric_cols_df1 if c not in key_like_cols]
pollutant_cols_df2 = [c for c in numeric_cols_df2 if c not in key_like_cols]

pollutant_cols = sorted(list(set(pollutant_cols_df1).intersection(set(pollutant_cols_df2))))

print("Number of pollutant columns to compare/adjust:", len(pollutant_cols))

Merged df1 and df2:
  df1 rows: 912
  df2 rows: 912
  merged rows (should be close to or equal to df1/df2): 912
Number of pollutant columns to compare/adjust: 74


In [9]:
for col in pollutant_cols:
    col_df1 = f"{col}_df1"
    col_df2 = f"{col}_df2"
    # Create a correction column representing (df1 - df2) for this pollutant
    merged_12[f"{col}_corr"] = merged_12[col_df1] - merged_12[col_df2]

for col in pollutant_cols:
    col_df1 = f"{col}_df1"
    col_df2 = f"{col}_df2"
    merged_12[f"{col}_diff"] = merged_12[col_df2] - merged_12[col_df1]  # df2 - df1 
    # Guard against divide-by-zero for percent difference:
    denom = merged_12[col_df1].replace(0, np.nan)
    merged_12[f"{col}_percdiff"] = (merged_12[col_df2] - merged_12[col_df1]) / denom * 100.0

diff_cols = [f"{c}_diff" for c in pollutant_cols]
if diff_cols:
    merged_12['max_abs_diff_value'] = merged_12[diff_cols].abs().max(axis=1)
    merged_12['max_abs_diff_column'] = merged_12[diff_cols].abs().idxmax(axis=1)

corr_cols = [f"{c}_corr" for c in pollutant_cols]
corr_with_keys = merged_12[align_keys + corr_cols].copy()

C:\Users\x12la\AppData\Local\Temp\ipykernel_19704\2151277604.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_12[f"{col}_diff"] = merged_12[col_df2] - merged_12[col_df1]  # df2 - df1
C:\Users\x12la\AppData\Local\Temp\ipykernel_19704\2151277604.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  merged_12[f"{col}_percdiff"] = (merged_12[col_df2] - merged_12[col_df1]) / denom * 100.0
C:\Users\x12la\AppData\Local\Temp\ipykernel_19704\2151277604.py:10: PerformanceWarning: DataFrame is highly fragmented.  This is usually 

In [10]:
df3_adjusted = pd.merge(
    df3,
    corr_with_keys,
    on=align_keys,
    how='left'  # left: keep every df3 row; if a correction is missing, we'll treat it as 0
)

for c in corr_cols:
    df3_adjusted[c] = df3_adjusted[c].fillna(0.0)

for col in pollutant_cols:
    corr_col = f"{col}_corr"
    # df3_adjusted[col] is the original df3 value; subtract the correction to remove df1->df2 bias
    df3_adjusted[col] = df3_adjusted[col] - df3_adjusted[corr_col]
    df3_adjusted[col] = df3_adjusted[col].clip(lower=0.0)

df3_adjusted.drop(columns=corr_cols, inplace=True)
print("Applied corrections to df3 to create df3_adjusted.")

Applied corrections to df3 to create df3_adjusted.


In [11]:
df4 = df2.copy()

df1['veh_code'] = _veh_code_from_agg_scc(df1['agg_scc'])
df2['veh_code'] = _veh_code_from_agg_scc(df2['agg_scc'])
df3_adjusted['veh_code'] = _veh_code_from_agg_scc(df3_adjusted['agg_scc'])
df4['veh_code'] = _veh_code_from_agg_scc(df4['agg_scc'])

target_veh_codes = {'52','53','61','62'}

df4_indexed = df4.set_index(align_keys).sort_index()
df3a_indexed = df3_adjusted.set_index(align_keys).sort_index()

mask_target = df4['veh_code'].isin(target_veh_codes)
keys_to_replace = df4.loc[mask_target, align_keys]

print("Rows in df4 with vehicle codes to replace (52,53,61,62):", len(keys_to_replace))

idx_df4 = pd.MultiIndex.from_frame(keys_to_replace)
common_index = idx_df4.intersection(df3a_indexed.index)

print("Rows that actually match in df3_adjusted for replacement:", len(common_index))

Rows in df4 with vehicle codes to replace (52,53,61,62): 285
Rows that actually match in df3_adjusted for replacement: 285


In [12]:
shared_pollutant_cols = [c for c in pollutant_cols if c in df4_indexed.columns and c in df3a_indexed.columns]

df4_indexed.loc[common_index, shared_pollutant_cols] = df3a_indexed.loc[common_index, shared_pollutant_cols].values

df4 = df4_indexed.reset_index()
print("df4 has been created from df2 with selected rows replaced by df3_adjusted for vehicle codes 52, 53, 61, 62.")


df4 has been created from df2 with selected rows replaced by df3_adjusted for vehicle codes 52, 53, 61, 62.


In [13]:
df4.loc[(df4['agg_scc'].astype(str).str.zfill(10).str[4:6].isin(["62"]))]

,MOVESScenarioID,yearID,FIPS,monthID,agg_scc,temperature,THC_INV,CO_INV,NOX,CH4_INV,...,HONO,NAPHTH,NAPHTHALENE,NH3,NO,NO2,PMC,PMFINE,SO2,veh_code
437,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,25,1.234249,15.999708,39.614344,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381,62
438,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,30,1.234249,15.999707,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381,62
439,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,35,1.234249,15.999709,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011790,0.007419,0.000381,62
440,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,40,1.234249,15.999707,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011786,0.007419,0.000381,62
441,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,45,1.234249,15.999709,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011790,0.007419,0.000381,62
442,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,50,1.234249,15.999708,39.614344,0.310971,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381,62
443,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,55,1.234249,15.999709,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011790,0.007418,0.000381,62
444,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,60,1.234249,15.999708,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011790,0.007418,0.000381,62
445,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,65,1.234249,15.999708,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011786,0.007419,0.000381,62
446,rd_17031_2022_7_t25_115,2022,17031,7,2202620192,70,1.234249,15.999708,39.614345,0.310970,...,0.006890,0.000544,4.246969e-06,0.024949,0.692004,0.162322,0.011789,0.007419,0.000381,62


In [14]:
if 'veh_code' in df4.columns:
    df4 = df4.drop(columns=['veh_code'])

In [15]:
df2_source_path = "Baseline_EFs/rateperhouroni_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_7.csv"

# read metadata lines (you already do this)
meta_lines = []
with open(df2_source_path, 'r', encoding='utf-8', newline='') as f_meta:
    meta_lines.append(next(f_meta))
    meta_lines.append(next(f_meta))
 

df2_text = pd.read_csv(
    df2_source_path,
    skiprows=2,
    dtype=str,                # keep as text
    low_memory=False
)

assert len(df2_text) == len(df2), "Row count mismatch between df2_text and df2"

In [16]:
def format_like(original: str, value: float) -> str:
    """
    Format `value` to match the style of `original`:
    - respects scientific notation vs fixed vs integer
    - respects number of decimal places
    """
    if pd.isna(value):
        # If original is blank, keep it blank; otherwise just return original
        return "" if (original is None or str(original).strip() == "") else str(original)

    if original is None:
        original = ""
    original = str(original).strip()

    # If original was blank, fall back to a generic but non-evil format
    if original == "":
        # choose a reasonable default; adjust if you want
        return f"{value:.6f}".rstrip("0").rstrip(".") or "0"

    # Scientific notation?
    if "e" in original or "E" in original:
        lower = "e" in original and "E" not in original
        # Split off the base to count decimals
        base_part = re.split("[eE]", original, maxsplit=1)[0]
        if "." in base_part:
            dec = len(base_part.split(".")[-1])
        else:
            dec = 0
        fmt_spec = f".{dec}{'e' if lower else 'E'}"
        return format(value, fmt_spec)

    # Fixed-point decimal?
    if "." in original:
        dec = len(original.split(".")[-1])
        fmt_spec = f".{dec}f"
        return format(value, fmt_spec)

    # Otherwise, treat as integer
    return format(value, ".0f")

In [17]:
df2_keys = df2[align_keys].copy()
df2_keys["_row_id"] = np.arange(len(df2_keys))

# 2) Subset df4 to align_keys + pollutant columns you actually modified
#    `shared_pollutant_cols` was already defined earlier in your script.
df4_small = df4[align_keys + shared_pollutant_cols].copy()

# 3) Merge to get new numeric values for each row in df2, if available
df2_with_new = df2_keys.merge(df4_small, on=align_keys, how="left")

# 4) Restore original row order
df2_with_new = df2_with_new.sort_values("_row_id").set_index("_row_id")

# 5) Identify which rows are target vehicle codes (52, 53, 61, 62)
df2_with_new["veh_code"] = _veh_code_from_agg_scc(df2_with_new["agg_scc"])
mask_target = df2_with_new["veh_code"].isin(target_veh_codes)

# --- Build output strings DataFrame starting from original text ---

out_df = df2_text.copy()  # all columns as strings; correct order & header

# Ensure we only touch pollutant columns that exist in df2_text
pollutant_cols_to_write = [c for c in shared_pollutant_cols if c in out_df.columns]

# Loop row-by-row for target rows, column-by-column for pollutant columns
# and format each new numeric value to mimic the original string pattern.
for col in pollutant_cols_to_write:
    # new numeric values (may be NaN where df4 had no data — e.g. non-target rows)
    new_vals = df2_with_new[col].to_numpy()
    # current string values from original file
    current_strings = out_df[col].astype(str).to_numpy()

    for i in range(len(out_df)):
        if not mask_target.iloc[i]:
            # Not one of the rows we replaced; keep original string
            continue

        val = new_vals[i]
        # If we somehow don't have a new numeric value, keep original
        if pd.isna(val):
            continue

        orig_str = current_strings[i]
        current_strings[i] = format_like(orig_str, val)

    out_df[col] = current_strings


# --- Write out new CSV with original metadata lines and formatting preserved ---

out_path = "Combined_EFs/rateperhouroni_smoke_aq_cb6_saprc_20240220_rates2022v1-2022-20240214_17031_7.csv"

with open(out_path, "w", encoding="utf-8", newline="") as f_out:
    # write the two original metadata lines verbatim
    for line in meta_lines:
        f_out.write(line if line.endswith("\n") else line + "\n")

    # write the data; everything is already correctly formatted as strings
    out_df.to_csv(
        f_out,
        index=False,
        quoting=csv.QUOTE_MINIMAL
    )